In [2]:
import re
import os 
import csv
import json
import pandas as pd

# klines 디렉토리 내의 파일 탐색
klines_file_path = "../data/binance/futures/um/monthly/klines/BTCUSDT/30m"
klines_use_columns = ["open_time", "open", "high", "low", "close", "volume", "quote_volume", "count", "taker_buy_volume", "taker_buy_quote_volume"]
klines_file_list = os.listdir(klines_file_path)
klines_file_list.sort()

# position distribution(pd) 디렉토리 내의 파일 탐색
pd_file_path = "../data/binance/futures/um/monthly/position_distribution/1800000"
pd_file_list = os.listdir(pd_file_path)
pd_file_list.sort()

# 저장할 데이터 프레임 생성
result_path = "../data/binance/futures/um/dataset/"
result_df = None

for pd_file in pd_file_list:
    # pd_df 읽기
    pd_df = pd.read_csv(os.path.join(pd_file_path, pd_file), index_col=0)
    
    # JSON 형태로 저장된 리스트 컬럼 변환
    pd_df['retail_long'] = pd_df['retail_long'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    pd_df['retail_short'] = pd_df['retail_short'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    pd_df['institutional_long'] = pd_df['institutional_long'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    pd_df['institutional_short'] = pd_df['institutional_short'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

    # 날짜/시간 열을 인덱스로 설정
    pd_df.index = pd.to_datetime(pd_df.index)
    
    # pd_df file 이름에서 년도, 월 추출
    match = re.search(r"BTCUSDT-pd-(\d{4})-(\d{2})\.csv", pd_file)
    if not match:
        print(f"파일명 형식이 맞지 않습니다: {pd_file}")
        continue
    year, month = match.groups()
    
    # 년도, 월에 따라서 klines_file_list에서 해당 파일 찾기
    klines_file = [file for file in klines_file_list if file.endswith(f"{year}-{month}.csv")][0]
    klines_df = pd.read_csv(os.path.join(klines_file_path, klines_file), index_col=0, usecols=klines_use_columns)
    
    # open_time을 datetime으로 변환, 30분 뒤로 밀기
    klines_df.index = pd.to_datetime(klines_df.index, unit="ms")
    klines_df.index = klines_df.index + pd.Timedelta(minutes=30)
    # klines_df와 pd_df 결합
    combined_df = pd.concat([klines_df, pd_df], axis=1)
    
    if result_df is None:
        result_df = combined_df
    else:
        result_df = pd.concat([result_df, combined_df])
    
# 9:1 비율로 train/test 데이터 분리
ratio = 0.9 
train_df = result_df.iloc[:int(len(result_df) * ratio)]
test_df = result_df.iloc[int(len(result_df) * ratio):]

# train_df와 test_df를 각각 CSV 파일로 저장
train_df.to_csv(os.path.join(result_path, "train.csv"), index=True, quoting=csv.QUOTE_NONNUMERIC)
test_df.to_csv(os.path.join(result_path, "test.csv"), index=True, quoting=csv.QUOTE_NONNUMERIC)